**Pydantic +Parsing**


In [16]:
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv
from openai import OpenAI
import json

In [9]:
load_dotenv()
client = OpenAI()

In [10]:

message = [{"role": "system", "content": "Answer the the user question in the format of content, conficdence and sources"},
           {"role": "user", "content": "How is the growth rate of India?"}]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=message)
##note that the response is free text but it has identifiers for confidence and sources. This can be improved. See below
print(response.choices[0].message.content)

India's growth rate has been fluctuating in recent years, but it has generally been experiencing positive growth. According to the World Bank, India's GDP growth rate was 4.2% in 2019 and is projected to be 5.4% in 2020. The International Monetary Fund (IMF) predicts that India's growth rate will further increase to 6.1% in 2021. These figures indicate a moderate growth trajectory for the Indian economy.

Content: The growth rate of India has been positive but fluctuating in recent years.
Confidence: Moderate
Sources: World Bank, International Monetary Fund


In [20]:

class Answer(BaseModel):
    # the description is meant for LLM not the developer.
    content: str = Field(..., description="The main answwer content") 
    confidence: float = Field(..., description="The confidence level of the answer", ge=0.0, le=1.0)
    sources: list[str] = Field(..., description="A list of sources that support the answer")

In [ ]:
message = [{"role": "system", "content": '''Answer the the user question in the JSON format which contains keys like content, conficdence 
(score bw 0-1) and sources (reference source) 
Sample ouput: {content: <real response>, 
confidence: <score bw 0-1>,
 sources: [source1, source2]}'''},
           {"role": "user", "content": "How is the growth rate of India?"}]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=message)
##note that the response is now a dictionary similar to what a json structure resopnse will be. 
#When no sample output was given, it still returned a dictionary structure with small some differences. The sources was not a list.
## NOTE: We are still using the prompt and not using a pydantic structure. 
# We can evaluate the output using the use the pydantic structure to validate the output. 
# This is a good way to ensure that the output is in the desired format. SEE BELOW
print(response.choices[0].message.content)

{
    "content": "As of 2021, India's GDP growth rate is estimated to be around 9.5%. However, due to various factors such as the impact of the COVID-19 pandemic, the actual growth rate may vary. India is considered one of the fastest-growing major economies in the world.",
    "confidence": 0.85,
    "sources": ["World Bank", "IMF"]
}


In [21]:

class Answer(BaseModel):
    # the description is meant for LLM not the developer.
    content: str = Field(..., description="The main answwer content") 
    confidence: float = Field(..., description="The confidence level of the answer", ge=0.0, le=1.0)
    sources: list[str] = Field(..., description="A list of sources that support the answer")

In [ ]:
message = [{"role": "system", "content": '''Answer the the user question in the JSON format which contains keys like content, conficdence 
(score bw 0-1) and sources (reference source) 
Sample ouput: {content: <real response>, 
confidence: <score bw 0-1>,
 sources: [source1, source2]}'''},
           {"role": "user", "content": "How is the growth rate of India?"}]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=message)
output = response.choices[0].message.content
json_output = json.loads(output)
ans = Answer.model_validate(json_output)
print(ans)
#NOTE: The above code is a good way to ensure that the output is in the desired format.
# When the class had name mismatch with the output or data type mismatch, it throws a validattion error
#When the class definition had an attribute called "answer" insterad of "content", the evaluator returned a validation error.
# we are not passing the pydantic structure to the LLM. We are only evaluating by tweaking the prompt to ensure that the output is in the desired format.


content='The growth rate of India has been fluctuating over the years. In recent times, it has experienced a slowdown due to various economic factors. However, the long-term growth trajectory of India remains positive, and the country is expected to bounce back in the coming years.' confidence=0.85 sources=['https://www.worldbank.org/en/news/feature/2021/10/07/india-economic-update-october-2021', 'https://www.reuters.com/world/india/reserve-bank-india-sees-2021-22-gdp-growth-94-2022-23-uk-sinha-2021-12-03/']


In [22]:
message = [{"role": "system", "content": '''Answer the the user question in the JSON format which contains keys like content, conficdence 
(score bw 0-1) and sources (reference source) 
Sample ouput: {content: <real response>, 
confidence: <score bw 0-1>,
 sources: [source1, source2]}'''},
           {"role": "user", "content": "How is the growth rate of India?"}]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=message,
    response_format = {"type": "json_object"})
output = response.choices[0].message.content
json_output = json.loads(output)
ans = Answer.model_validate(json_output)
#NOTE: using response_format = {"type": "json_object"} is a good way to ensure that the output is in the desire json format. 
# We re still not using the pydantic structure to generate the ouput.
print(ans)

content="The growth rate of India has been fluctuating in recent years but has generally been positive. As of 2021, India's GDP growth rate was estimated to be around 9.2%. However, due to various factors such as the COVID-19 pandemic and economic reforms, the growth rate can vary." confidence=0.85 sources=['https://www.investopedia.com/indian-gdp-growth-rate-statistics-4773321', 'https://tradingeconomics.com/india/gdp-growth']


In [46]:
#TOOL enforcement on LLM to ensure that the output is in the desired format.
tool = [
    {
        "type": "function",
        "function": {
            "name": "answer_question",
        "description": "provide answer to the user's question",
        "parameters":{
            "type": "object",
            "properties": {
                "content":
                    {
                        "type": "string",
                        "description": "The main answwer content"
                    },
                "confidence":
                    {
                        "type": "number",
                        "description": "The confidence level of the answer",
                        "minimum": 0.0,
                        "maximum": 1.0
                    }, 
                "sources":
                    {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "A list of sources that support the answer",
                       
                    }
            },
                            },
        "required": ["content", "confidence", "sources"]
                    }
    }

]

In [ ]:
message = [{"role": "user", "content": "How is the growth rate of India?"}]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=message,
    tools= tool,
    tool_choice={"type": "function", "function": {"name": "answer_question"}})
tool_call = response.choices[0].message.tool_calls[0]
output = tool_call.function.arguments
json_output = json.loads(output)
ans = Answer.model_validate(json_output)
#NOTE: The tool structure is enforcing the output to be in the desired format. The LLM is not free to generate any output. It has to follow the tool structure.
# We are using the tool call output.
print(ans)

content="The growth rate of India has been fluctuating in recent years. In 2021, India's GDP growth rate was estimated to be around 10.1%. However, the growth rate may vary due to various factors like government policies, global economic conditions, and other external factors." confidence=0.85 sources=['https://www.focus-economics.com/countries/india']
